In [8]:
import itertools
import shutil
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

from polars_ml import Pipeline

out_dir = Path("out")
out_dir.mkdir(exist_ok=True, parents=True)

pl.Config.set_tbl_rows(100)

train_df = pl.read_csv("../../data/raw/train.csv")
test_df = pl.read_csv("../../data/raw/test.csv")
submit_df = pl.read_csv("../../data/raw/sample_submission.csv")

train_df = train_df.drop("id").cast({"date": pl.Date})
test_df = test_df.drop("id").cast({"date": pl.Date})

gdp_df = pl.read_csv("../../data/external/gdp.csv")

countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()
years = [2010 + i for i in range(11)]

train_adj_df = pl.read_parquet("../../data/processed/train_num_sold_adj.parquet")
pp = Pipeline().standard_scale(
    "num_sold_adj", by="country", component_name="num_sold_adj_scale_by_country"
)
train_adj_df = pp.fit_transform(train_adj_df)
train_adj_df.head()

date,country,store,product,num_sold,year,month,week,weekday,day,gdp,coef,gdp_linear_pred,num_sold_adj
date,str,str,str,f64,i32,i8,i8,i8,i8,f64,f64,f64,f64
2010-01-01,"""Canada""","""Discount Stickers""","""Holographic Goose""",null,2010,1,53,5,1,47560.666601,0.002052,97.591677,null
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle""",973.0,2010,1,53,5,1,47560.666601,0.014726,700.374214,2.235554
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle Tiers""",906.0,2010,1,53,5,1,47560.666601,0.012158,578.219414,2.685077
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler""",423.0,2010,1,53,5,1,47560.666601,0.006636,315.611755,0.888832
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler Dark Mode""",491.0,2010,1,53,5,1,47560.666601,0.007769,369.48372,1.003979


In [6]:
from polars_ml.utils import get_country_holidays

holidays_df = get_country_holidays(countries, years)
holidays_df.head()

date,holiday,country
date,str,str
2016-01-01,"""New Year's Day""","""Kenya"""
2016-03-25,"""Good Friday""","""Kenya"""
2016-03-28,"""Easter Monday""","""Kenya"""
2016-05-01,"""Labour Day""","""Kenya"""
2016-05-02,"""Labour Day (observed)""","""Kenya"""


In [15]:
(
    train_adj_df.join(holidays_df, on=["country", "date"], how="left")
    .with_columns(pl.col("holiday").is_not_null().cast(pl.UInt8))
    .filter(
        (pl.col("country") == "Canada")
        & (pl.col("store") == "Discount Stickers")
        & (pl.col("product") == "Kaggle")
    )
    .with_columns(
        [
            pl.col("holiday")
            .shift(i)
            .over(["country", "store", "product"])
            .alias(f"holiday_{i}")
            for i in range(-2, 3)
        ]
    )
    .drop_nulls()
    .head()
)

date,country,store,product,num_sold,year,month,week,weekday,day,gdp,coef,gdp_linear_pred,num_sold_adj,holiday,holiday_-10,holiday_-9,holiday_-8,holiday_-7,holiday_-6,holiday_-5,holiday_-4,holiday_-3,holiday_-2,holiday_-1,holiday_0,holiday_1,holiday_2,holiday_3,holiday_4,holiday_5,holiday_6,holiday_7,holiday_8,holiday_9,holiday_10
date,str,str,str,f64,i32,i8,i8,i8,i8,f64,f64,f64,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle""",973.0,2010,1,53,5,1,47560.666601,0.014726,700.374214,2.235554,1,0,0,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null,null,null,null,null
2010-01-02,"""Canada""","""Discount Stickers""","""Kaggle""",881.0,2010,1,53,6,2,47560.666601,0.014726,700.374214,1.485734,0,0,0,0,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null,null,null,null
2010-01-03,"""Canada""","""Discount Stickers""","""Kaggle""",1003.0,2010,1,53,7,3,47560.666601,0.014726,700.374214,2.48006,0,0,0,0,0,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null,null,null
2010-01-04,"""Canada""","""Discount Stickers""","""Kaggle""",744.0,2010,1,1,1,4,47560.666601,0.014726,700.374214,0.369154,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null,null
2010-01-05,"""Canada""","""Discount Stickers""","""Kaggle""",707.0,2010,1,1,2,5,47560.666601,0.014726,700.374214,0.067596,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null


In [ ]:
start_date = datetime(2010, 1, 1)

train_adj_df = (
    train_adj_df.with_columns(
        (pl.col("date") - start_date).dt.total_days().alias("ordinal_day")
    )
    .with_columns(
        pl.col("ordinal_day").mul(2 * np.pi / 365).sin().alias("sin_period_year"),
        pl.col("ordinal_day").mul(2 * np.pi / 365).cos().alias("cos_period_year"),
        pl.col("ordinal_day")
        .mul(2 * np.pi / 365 / 2)
        .sin()
        .alias("sin_period_2_years"),
        pl.col("ordinal_day")
        .mul(2 * np.pi / 365 / 2)
        .cos()
        .alias("cos_period_2_years"),
        ((pl.col("day") - 1) // 7 + 1).alias("week_of_month"),
    )
    .join(holidays_df, on=["country", "date"], how="left")
    .with_columns(pl.col("holiday").is_not_null().cast(pl.UInt8))
    .with_columns(
        [
            pl.col("holiday")
            .shift(i)
            .over(["country", "store", "product"])
            .alias(f"holiday_{i}")
            for i in range(-2, 3)
        ]
    )
    .drop_nulls()
)
tmp = (
    train_adj_df.drop(
        "date",
        "num_sold",
        "year",
        "month",
        "week",
        # "weekday",
        "day",
        "week_of_month",
        "gdp",
        "coef",
        "gdp_linear_pred",
        "ordinal_day",
    )
    .to_dummies(
        ["country", "store", "product", "weekday"]
        + [f"holiday_{i}" for i in range(-2, 3)]
    )
    .with_columns(
        (pl.col("num_sold_adj") - pl.col("num_sold_adj").mean())
        / pl.col("num_sold_adj").std()
    )
)
tmp = Pipeline().polynomial(cs.exclude("num_sold_adj"), degree=2).transform(tmp)  # type: ignore
X = tmp.drop_nulls().drop("num_sold_adj").to_numpy()
y = tmp.drop_nulls()["num_sold_adj"].to_numpy()

model = LinearRegression()
model.fit(X, y)
